# MTEX-Style Raster + ML EBSD Stitching

This notebook stitches EBSD `.ang` tiles using a map-first workflow: gridify raster EBSD maps, register IQ/CI overlap strips, score candidate shifts with FCC orientation consistency, then train a small self-supervised ML scorer to rank seam translations. Grain reconstruction should be done after this stitched map step.


## Optional Conda / Kernel Setup

Use this if the active notebook kernel does not already have the dependencies. The commands are left commented so the notebook is safe to run in an existing environment.

In [ ]:
# Optional conda setup. Run these commands in a terminal if this notebook kernel
# does not already have the dependencies installed.
#
# conda create -n ebsd-stitch python=3.11 -y
# conda activate ebsd-stitch
# python -m pip install -r requirements-ebsd-stitching.txt
# python -m ipykernel install --user --name ebsd-stitch --display-name "Python (ebsd-stitch)"
#
# In Jupyter, switch the kernel to: Python (ebsd-stitch)

import os
from pathlib import Path
cache_root = Path('.cache')
os.environ.setdefault('MPLCONFIGDIR', str(cache_root / 'matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(cache_root))
os.environ.setdefault('NUMBA_CACHE_DIR', str(cache_root / 'numba'))
print('Using local cache root:', cache_root.resolve())


## Imports and Global Constants

In [ ]:
import argparse
import json
import math
import os
import re

os.environ.setdefault('MPLCONFIGDIR', str(Path('.cache') / 'matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(Path('.cache')))
os.environ.setdefault('NUMBA_CACHE_DIR', str(Path('.cache') / 'numba'))
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

os.environ.setdefault("MPLCONFIGDIR", str(Path(".cache") / "matplotlib"))

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation
from skimage.registration import phase_cross_correlation
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


SEED = 521
RNG = np.random.default_rng(SEED)


ANG_COLUMNS = [
    "phi1",
    "Phi",
    "phi2",
    "x",
    "y",
    "IQ",
    "CI",
    "phase",
    "SEM_signal",
    "fit",
]


## Tile Data Structures

In [ ]:
@dataclass
class TileGrid:
    name: str
    row: int
    col: int
    path: Path
    x_values: np.ndarray
    y_values: np.ndarray
    x_step: float
    y_step: float
    phi1: np.ndarray
    Phi: np.ndarray
    phi2: np.ndarray
    iq: np.ndarray
    ci: np.ndarray
    phase: np.ndarray
    valid: np.ndarray

    @property
    def shape(self) -> Tuple[int, int]:
        return self.iq.shape

    @property
    def height(self) -> int:
        return self.iq.shape[0]

    @property
    def width(self) -> int:
        return self.iq.shape[1]


@dataclass
class PairCandidate:
    pair_id: str
    direction: str
    tile_a: str
    tile_b: str
    shift_y: int
    shift_x: int
    origin_dy: int
    origin_dx: int
    iq_corr: float
    ci_corr: float
    valid_fraction: float
    median_misorientation_deg: float
    p90_misorientation_deg: float
    overlap_pixels: int
    candidate_source: str
    classical_score: float
    pseudo_label: int
    ml_probability: float = float("nan")
    final_score: float = float("nan")
    selected: bool = False
    accepted: bool = False


## ANG Loading and MTEX-Style Gridify

In [ ]:
def read_ang_numeric(path: Path) -> pd.DataFrame:
    """Read numeric ANG rows into a dataframe, preserving common columns."""
    numeric_rows: List[List[float]] = []
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            parts = stripped.split()
            try:
                numeric_rows.append([float(p) for p in parts])
            except ValueError:
                continue

    if not numeric_rows:
        raise ValueError(f"No numeric ANG rows found in {path}")

    ncols = max(len(row) for row in numeric_rows)
    arr = np.full((len(numeric_rows), ncols), np.nan, dtype=float)
    for i, row in enumerate(numeric_rows):
        arr[i, : len(row)] = row

    cols = ANG_COLUMNS[:ncols] + [f"extra_{i}" for i in range(max(0, ncols - len(ANG_COLUMNS)))]
    df = pd.DataFrame(arr, columns=cols[:ncols])
    for col, default in [("IQ", 1.0), ("CI", 1.0), ("phase", 1)]:
        if col not in df:
            df[col] = default

    # ANG files are normally radians. If they look degree-like, convert.
    if df[["phi1", "Phi", "phi2"]].to_numpy(dtype=float).max() > 2 * np.pi + 1e-3:
        df[["phi1", "Phi", "phi2"]] = np.deg2rad(df[["phi1", "Phi", "phi2"]])

    df["phase"] = 1  # SCC 316 is treated as single-phase austenitic FCC.
    return df


def gridify_ang(df: pd.DataFrame, name: str, row: int, col: int, path: Path) -> TileGrid:
    """MTEX-style gridify: convert unordered EBSD points into regular arrays."""
    xs = np.sort(df["x"].dropna().unique())
    ys = np.sort(df["y"].dropna().unique())
    rectangular_fill = len(df) / max(len(xs) * len(ys), 1)
    row_counts = df.groupby("y").size()
    use_scan_rows = rectangular_fill < 0.75 and row_counts.nunique() <= 3

    if use_scan_rows:
        # Tilt-corrected or hex-like ANG exports can have staggered x positions:
        # unique x/y would create a half-empty grid. MTEX gridify preserves scan
        # rows, so use sorted points within each y row as raster columns.
        width = int(row_counts.mode().iloc[0])
        shape = (len(ys), width)
        arrays = {
            "phi1": np.full(shape, np.nan, dtype=float),
            "Phi": np.full(shape, np.nan, dtype=float),
            "phi2": np.full(shape, np.nan, dtype=float),
            "iq": np.full(shape, np.nan, dtype=float),
            "ci": np.full(shape, np.nan, dtype=float),
            "phase": np.full(shape, 1, dtype=int),
        }
        x_rows = []
        for yi, (_, gdf) in enumerate(df.sort_values(["y", "x"]).groupby("y", sort=True)):
            gdf = gdf.sort_values("x").head(width)
            n = len(gdf)
            arrays["phi1"][yi, :n] = gdf["phi1"].to_numpy()
            arrays["Phi"][yi, :n] = gdf["Phi"].to_numpy()
            arrays["phi2"][yi, :n] = gdf["phi2"].to_numpy()
            arrays["iq"][yi, :n] = gdf["IQ"].to_numpy()
            arrays["ci"][yi, :n] = gdf["CI"].to_numpy()
            arrays["phase"][yi, :n] = 1
            if n == width:
                x_rows.append(gdf["x"].to_numpy())
        x_values = np.nanmedian(np.vstack(x_rows), axis=0) if x_rows else np.arange(width, dtype=float)
    else:
        x_index = {v: i for i, v in enumerate(xs)}
        y_index = {v: i for i, v in enumerate(ys)}
        shape = (len(ys), len(xs))
        arrays = {
            "phi1": np.full(shape, np.nan, dtype=float),
            "Phi": np.full(shape, np.nan, dtype=float),
            "phi2": np.full(shape, np.nan, dtype=float),
            "iq": np.full(shape, np.nan, dtype=float),
            "ci": np.full(shape, np.nan, dtype=float),
            "phase": np.full(shape, 1, dtype=int),
        }

        for rec in df.itertuples(index=False):
            xi = x_index[getattr(rec, "x")]
            yi = y_index[getattr(rec, "y")]
            arrays["phi1"][yi, xi] = getattr(rec, "phi1")
            arrays["Phi"][yi, xi] = getattr(rec, "Phi")
            arrays["phi2"][yi, xi] = getattr(rec, "phi2")
            arrays["iq"][yi, xi] = getattr(rec, "IQ")
            arrays["ci"][yi, xi] = getattr(rec, "CI")
            arrays["phase"][yi, xi] = 1
        x_values = xs

    valid = np.isfinite(arrays["phi1"]) & np.isfinite(arrays["iq"])
    x_step = float(np.nanmedian(np.diff(x_values))) if len(x_values) > 1 else 1.0
    y_step = float(np.nanmedian(np.diff(ys))) if len(ys) > 1 else 1.0
    return TileGrid(
        name=name,
        row=row,
        col=col,
        path=path,
        x_values=x_values,
        y_values=ys,
        x_step=x_step,
        y_step=y_step,
        phi1=arrays["phi1"],
        Phi=arrays["Phi"],
        phi2=arrays["phi2"],
        iq=arrays["iq"],
        ci=arrays["ci"],
        phase=arrays["phase"],
        valid=valid,
    )


## FCC Symmetry and Orientation Math

In [ ]:
def _proper_cubic_symmetry_operators() -> np.ndarray:
    from itertools import permutations, product

    ops = []
    for perm in permutations(range(3)):
        p = np.zeros((3, 3), dtype=float)
        for row, col in enumerate(perm):
            p[row, col] = 1.0
        for signs in product([-1.0, 1.0], repeat=3):
            s = np.diag(signs) @ p
            if np.linalg.det(s) > 0.5:
                ops.append(s)
    return np.stack(ops)


CUBIC_SYMMETRY = _proper_cubic_symmetry_operators()


def euler_to_matrix(phi1: np.ndarray, Phi: np.ndarray, phi2: np.ndarray) -> np.ndarray:
    """Bunge-like Euler angles to rotation matrices, matching the existing notebook."""
    c1, s1 = np.cos(phi1), np.sin(phi1)
    c, s = np.cos(Phi), np.sin(Phi)
    c2, s2 = np.cos(phi2), np.sin(phi2)
    out = np.empty(phi1.shape + (3, 3), dtype=float)
    out[..., 0, 0] = c1 * c2 - s1 * s2 * c
    out[..., 0, 1] = s1 * c2 + c1 * s2 * c
    out[..., 0, 2] = s2 * s
    out[..., 1, 0] = -c1 * s2 - s1 * c2 * c
    out[..., 1, 1] = -s1 * s2 + c1 * c2 * c
    out[..., 1, 2] = c2 * s
    out[..., 2, 0] = s1 * s
    out[..., 2, 1] = -c1 * s
    out[..., 2, 2] = c
    return out


def cubic_misorientation_deg_many(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Approximate FCC/cubic disorientation using one-sided crystal symmetry.

    This is intentionally fast for raster scoring. It correctly collapses a
    180-degree cubic symmetry rotation to 0 degrees and is sufficient for seam
    ranking. Full two-sided disorientation can be added later if needed.
    """
    delta = np.einsum("nij,nkj->nik", a, b)
    best_trace = np.full(delta.shape[0], -3.0, dtype=float)
    for sym in CUBIC_SYMMETRY:
        sym_delta = np.einsum("ij,njk->nik", sym, delta)
        traces = np.einsum("nii->n", sym_delta)
        best_trace = np.maximum(best_trace, traces)
    cos_theta = np.clip((best_trace - 1.0) / 2.0, -1.0, 1.0)
    return np.rad2deg(np.arccos(cos_theta))


## Right/Down Overlap Registration and Candidate Scoring

In [ ]:
def normalize_image(img: np.ndarray) -> np.ndarray:
    out = img.astype(float).copy()
    finite = np.isfinite(out)
    if not finite.any():
        return np.zeros_like(out)
    med = np.nanmedian(out)
    out[~finite] = med
    lo, hi = np.nanpercentile(out, [1, 99])
    if hi <= lo:
        return np.zeros_like(out)
    out = np.clip((out - lo) / (hi - lo), 0.0, 1.0)
    return out


def pearson_corr(a: np.ndarray, b: np.ndarray, mask: np.ndarray) -> float:
    good = mask & np.isfinite(a) & np.isfinite(b)
    if good.sum() < 16:
        return 0.0
    av = a[good].astype(float)
    bv = b[good].astype(float)
    av -= av.mean()
    bv -= bv.mean()
    denom = np.linalg.norm(av) * np.linalg.norm(bv)
    return float(np.dot(av, bv) / denom) if denom > 1e-12 else 0.0


def shifted_slices(shape_a: Tuple[int, int], shape_b: Tuple[int, int], shift_y: int, shift_x: int):
    """Slices for comparing A to B after applying skimage-style shift to B."""
    ha, wa = shape_a
    hb, wb = shape_b
    ay0 = max(0, shift_y)
    ay1 = min(ha, hb + shift_y)
    ax0 = max(0, shift_x)
    ax1 = min(wa, wb + shift_x)
    if ay1 <= ay0 or ax1 <= ax0:
        return None
    by0 = ay0 - shift_y
    by1 = ay1 - shift_y
    bx0 = ax0 - shift_x
    bx1 = ax1 - shift_x
    return (slice(ay0, ay1), slice(ax0, ax1)), (slice(by0, by1), slice(bx0, bx1))


def strip_pair(a: TileGrid, b: TileGrid, direction: str, overlap_px: int):
    if direction == "horizontal":
        overlap_px = int(np.clip(overlap_px, 4, min(a.width, b.width)))
        return (
            {k: getattr(a, k)[:, -overlap_px:] for k in ["iq", "ci", "phi1", "Phi", "phi2", "valid"]},
            {k: getattr(b, k)[:, :overlap_px] for k in ["iq", "ci", "phi1", "Phi", "phi2", "valid"]},
        )
    overlap_px = int(np.clip(overlap_px, 4, min(a.height, b.height)))
    return (
        {k: getattr(a, k)[-overlap_px:, :] for k in ["iq", "ci", "phi1", "Phi", "phi2", "valid"]},
        {k: getattr(b, k)[:overlap_px, :] for k in ["iq", "ci", "phi1", "Phi", "phi2", "valid"]},
    )


def phase_corr_shift(ref: np.ndarray, moving: np.ndarray) -> Tuple[int, int]:
    ref_n = normalize_image(ref)
    mov_n = normalize_image(moving)
    shift, _, _ = phase_cross_correlation(ref_n, mov_n, upsample_factor=1, normalization=None)
    return int(round(float(shift[0]))), int(round(float(shift[1])))


def candidate_shift_set(strips_a: Dict[str, np.ndarray], strips_b: Dict[str, np.ndarray], radius: int) -> List[Tuple[int, int, str]]:
    bases = [(0, 0, "zero")]
    for channel in ["iq", "ci"]:
        try:
            sy, sx = phase_corr_shift(strips_a[channel], strips_b[channel])
            bases.append((sy, sx, f"{channel}_phase_corr"))
        except Exception:
            pass

    candidates: Dict[Tuple[int, int], str] = {}
    for sy, sx, source in bases:
        for dy in range(-radius, radius + 1):
            for dx in range(-radius, radius + 1):
                key = (sy + dy, sx + dx)
                candidates.setdefault(key, source)
    return [(sy, sx, source) for (sy, sx), source in sorted(candidates.items())]


def evaluate_candidate(
    pair_id: str,
    direction: str,
    tile_a: TileGrid,
    tile_b: TileGrid,
    strips_a: Dict[str, np.ndarray],
    strips_b: Dict[str, np.ndarray],
    shift_y: int,
    shift_x: int,
    source: str,
    overlap_px: int,
    orientation_samples: int,
) -> Optional[PairCandidate]:
    slices = shifted_slices(strips_a["iq"].shape, strips_b["iq"].shape, shift_y, shift_x)
    if slices is None:
        return None
    sa, sb = slices
    valid = strips_a["valid"][sa] & strips_b["valid"][sb]
    overlap_pixels = int(valid.sum())
    if overlap_pixels < 16:
        return None

    iq_corr = pearson_corr(strips_a["iq"][sa], strips_b["iq"][sb], valid)
    ci_corr = pearson_corr(strips_a["ci"][sa], strips_b["ci"][sb], valid)
    valid_fraction = float(overlap_pixels / valid.size)

    yy, xx = np.nonzero(valid)
    if len(yy) > orientation_samples:
        idx = RNG.choice(len(yy), size=orientation_samples, replace=False)
        yy, xx = yy[idx], xx[idx]

    a_phi1 = strips_a["phi1"][sa][yy, xx]
    a_Phi = strips_a["Phi"][sa][yy, xx]
    a_phi2 = strips_a["phi2"][sa][yy, xx]
    b_phi1 = strips_b["phi1"][sb][yy, xx]
    b_Phi = strips_b["Phi"][sb][yy, xx]
    b_phi2 = strips_b["phi2"][sb][yy, xx]
    finite_ori = np.isfinite(a_phi1) & np.isfinite(a_Phi) & np.isfinite(a_phi2) & np.isfinite(b_phi1) & np.isfinite(b_Phi) & np.isfinite(b_phi2)
    if finite_ori.sum() < 8:
        median_mis = 180.0
        p90_mis = 180.0
    else:
        ra = euler_to_matrix(a_phi1[finite_ori], a_Phi[finite_ori], a_phi2[finite_ori])
        rb = euler_to_matrix(b_phi1[finite_ori], b_Phi[finite_ori], b_phi2[finite_ori])
        mis = cubic_misorientation_deg_many(ra, rb)
        median_mis = float(np.nanmedian(mis))
        p90_mis = float(np.nanpercentile(mis, 90))

    orientation_score = math.exp(-median_mis / 8.0)
    shift_penalty = 0.015 * (abs(shift_y) + abs(shift_x))
    classical_score = (
        0.35 * max(iq_corr, 0.0)
        + 0.15 * max(ci_corr, 0.0)
        + 0.35 * orientation_score
        + 0.15 * valid_fraction
        - shift_penalty
    )

    pseudo_label = int(
        valid_fraction >= 0.60
        and iq_corr >= 0.45
        and median_mis <= 8.0
    )

    if direction == "horizontal":
        origin_dy = shift_y
        origin_dx = tile_a.width - overlap_px + shift_x
    else:
        origin_dy = tile_a.height - overlap_px + shift_y
        origin_dx = shift_x

    return PairCandidate(
        pair_id=pair_id,
        direction=direction,
        tile_a=tile_a.name,
        tile_b=tile_b.name,
        shift_y=shift_y,
        shift_x=shift_x,
        origin_dy=int(origin_dy),
        origin_dx=int(origin_dx),
        iq_corr=iq_corr,
        ci_corr=ci_corr,
        valid_fraction=valid_fraction,
        median_misorientation_deg=median_mis,
        p90_misorientation_deg=p90_mis,
        overlap_pixels=overlap_pixels,
        candidate_source=source,
        classical_score=float(classical_score),
        pseudo_label=pseudo_label,
    )


## Tile Manifest and Right/Down Pair Discovery

In [ ]:
def read_manifest(folder: Path) -> pd.DataFrame:
    manifest_path = folder / "tile_manifest.csv"
    if manifest_path.exists():
        manifest = pd.read_csv(manifest_path)
        manifest = manifest[manifest["tile_name"].astype(str).str.endswith("_clean.ang")].copy()
        if "distorted" in manifest:
            manifest = manifest[manifest["distorted"].astype(str).str.lower().isin(["false", "0", "no"])]
    else:
        rows = []
        for path in sorted(folder.glob("tile_r*_c*_clean.ang")):
            match = re.search(r"tile_r(\d+)_c(\d+)_clean\.ang$", path.name)
            if match:
                rows.append({"tile_name": path.name, "row": int(match.group(1)), "col": int(match.group(2))})
        manifest = pd.DataFrame(rows)

    if manifest.empty:
        raise ValueError(f"No clean tile manifest entries found under {folder}")

    manifest["path"] = manifest["tile_name"].map(lambda n: folder / str(n))
    manifest = manifest[manifest["path"].map(Path.exists)].copy()
    return manifest.sort_values(["row", "col"]).reset_index(drop=True)


def adjacent_pairs(manifest: pd.DataFrame) -> List[Tuple[pd.Series, pd.Series, str]]:
    by_rc = {(int(r.row), int(r.col)): r for _, r in manifest.iterrows()}
    pairs = []
    for (row, col), tile in sorted(by_rc.items()):
        right = by_rc.get((row, col + 1))
        if right is not None:
            pairs.append((tile, right, "horizontal"))
        down = by_rc.get((row + 1, col))
        if down is not None:
            pairs.append((tile, down, "vertical"))
    return pairs


def infer_overlap_px(a: TileGrid, b: TileGrid, row_a: pd.Series, row_b: pd.Series, direction: str) -> int:
    if direction == "horizontal" and {"xmin", "xmax"}.issubset(row_a.index):
        overlap_units = max(0.0, min(float(row_a.xmax), float(row_b.xmax)) - max(float(row_a.xmin), float(row_b.xmin)))
        return int(round(overlap_units / max(a.x_step, 1e-9)))
    if direction == "vertical" and {"ymin", "ymax"}.issubset(row_a.index):
        overlap_units = max(0.0, min(float(row_a.ymax), float(row_b.ymax)) - max(float(row_a.ymin), float(row_b.ymin)))
        return int(round(overlap_units / max(a.y_step, 1e-9)))
    return max(4, int(round((a.width if direction == "horizontal" else a.height) * 0.20)))


## Self-Supervised ML Seam Scorer and Placement Solver

In [ ]:
def train_ml_scorer(candidates: List[PairCandidate], use_training: bool = True, train_fraction: float = 0.70):
    """Train the self-supervised seam scorer on held-out-free pseudo-labels.

    The labels are generated from physics, not from manual annotations:
    candidates with high IQ/CI agreement and low FCC disorientation are
    pseudo-positives; poor candidates are pseudo-negatives. The model trains on
    a subset of seam pairs and predicts every candidate, including held-out
    seam pairs, so the validation report is not a same-candidate fit score.
    """
    from sklearn.metrics import average_precision_score, roc_auc_score

    feature_cols = [
        "iq_corr",
        "ci_corr",
        "valid_fraction",
        "median_misorientation_deg",
        "p90_misorientation_deg",
        "shift_abs",
        "classical_score",
    ]
    rows = []
    for c in candidates:
        rows.append(
            {
                "pair_id": c.pair_id,
                "iq_corr": c.iq_corr,
                "ci_corr": c.ci_corr,
                "valid_fraction": c.valid_fraction,
                "median_misorientation_deg": c.median_misorientation_deg,
                "p90_misorientation_deg": c.p90_misorientation_deg,
                "shift_abs": abs(c.shift_y) + abs(c.shift_x),
                "classical_score": c.classical_score,
                "pseudo_label": c.pseudo_label,
            }
        )
    df = pd.DataFrame(rows)

    def _split_metrics(split_name, mask, probs):
        y = df.loc[mask, "pseudo_label"].to_numpy(dtype=int)
        p = np.asarray(probs)[mask]
        out = {"n": int(mask.sum()), "positives": int(y.sum()), "negatives": int((1 - y).sum())}
        if len(np.unique(y)) > 1:
            out["roc_auc"] = float(roc_auc_score(y, p))
            out["pr_auc"] = float(average_precision_score(y, p))
        else:
            out["roc_auc"] = None
            out["pr_auc"] = None
        out["mean_probability"] = float(np.mean(p)) if len(p) else None
        return out

    report = {
        "mode": "self_supervised_logistic_regression" if use_training else "no_training_classical_score",
        "pseudo_label_rule": "positive when valid_fraction>=0.60, IQ_corr>=0.45, and median_FCC_misorientation<=8deg",
        "n_candidates": int(len(df)),
        "pseudo_positives": int(df["pseudo_label"].sum()) if not df.empty else 0,
        "pseudo_negatives": int((1 - df["pseudo_label"]).sum()) if not df.empty else 0,
    }

    if (not use_training) or df.empty or df["pseudo_label"].nunique() < 2 or df["pseudo_label"].sum() < 3:
        for c in candidates:
            c.ml_probability = float(np.clip(c.classical_score, 0.0, 1.0))
            c.final_score = c.ml_probability
            c.self_supervised_split = "not_trained"
        report["reason"] = "training disabled or insufficient pseudo-label diversity"
        return None, None, feature_cols, report

    pair_ids = np.array(sorted(df["pair_id"].unique()))
    rng = np.random.default_rng(SEED)
    rng.shuffle(pair_ids)
    n_train = int(np.clip(round(len(pair_ids) * train_fraction), 1, max(len(pair_ids) - 1, 1)))
    train_pairs = set(pair_ids[:n_train])
    train_mask = df["pair_id"].isin(train_pairs).to_numpy()
    val_mask = ~train_mask

    # If the pair-level split is degenerate, fall back to fitting all candidates
    # but keep the report honest.
    if df.loc[train_mask, "pseudo_label"].nunique() < 2 or df.loc[train_mask, "pseudo_label"].sum() < 3:
        train_mask = np.ones(len(df), dtype=bool)
        val_mask = np.zeros(len(df), dtype=bool)
        report["split_warning"] = "pair-level train split lacked both pseudo-label classes; fit on all candidates"

    x = df[feature_cols].to_numpy(dtype=float)
    y = df["pseudo_label"].to_numpy(dtype=int)
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x[train_mask])
    x_all = scaler.transform(x)
    model = LogisticRegression(class_weight="balanced", random_state=SEED, max_iter=1000)
    model.fit(x_train, y[train_mask])
    probs = model.predict_proba(x_all)[:, 1]

    for c, p, is_train, is_val in zip(candidates, probs, train_mask, val_mask):
        c.ml_probability = float(p)
        c.final_score = 0.65 * float(p) + 0.35 * float(np.clip(c.classical_score, 0.0, 1.0))
        c.self_supervised_split = "train" if is_train else "validation"

    report.update(
        {
            "train_fraction_by_pair": float(train_fraction),
            "n_train_pairs": int(len(set(df.loc[train_mask, "pair_id"]))),
            "n_validation_pairs": int(len(set(df.loc[val_mask, "pair_id"]))),
            "train": _split_metrics("train", train_mask, probs),
            "validation": _split_metrics("validation", val_mask, probs) if val_mask.any() else None,
        }
    )
    return model, scaler, feature_cols, report


def choose_best_candidates(
    candidates: List[PairCandidate],
    min_accept_score: float,
    max_accept_misorientation: float,
    min_valid_fraction: float,
) -> List[PairCandidate]:
    by_pair: Dict[str, List[PairCandidate]] = {}
    for c in candidates:
        by_pair.setdefault(c.pair_id, []).append(c)

    selected = []
    for pair_id, group in by_pair.items():
        best = max(group, key=lambda c: c.final_score)
        best.selected = True
        best.accepted = (
            best.final_score >= min_accept_score
            and best.median_misorientation_deg <= max_accept_misorientation
            and best.valid_fraction >= min_valid_fraction
        )
        selected.append(best)
    return selected


def solve_tile_origins(tiles: Dict[str, TileGrid], selected: List[PairCandidate]) -> Dict[str, Tuple[int, int]]:
    graph: Dict[str, List[Tuple[str, int, int]]] = {name: [] for name in tiles}
    for edge in selected:
        if not edge.accepted:
            continue
        graph[edge.tile_a].append((edge.tile_b, edge.origin_dy, edge.origin_dx))
        graph[edge.tile_b].append((edge.tile_a, -edge.origin_dy, -edge.origin_dx))

    start = min(tiles.values(), key=lambda t: (t.row, t.col)).name
    origins = {start: (0, 0)}
    queue = [start]
    while queue:
        current = queue.pop(0)
        cy, cx = origins[current]
        for nxt, dy, dx in graph[current]:
            if nxt in origins:
                continue
            origins[nxt] = (cy + dy, cx + dx)
            queue.append(nxt)

    # Disconnected fallback: place by manifest row/col pitch.
    horizontal_edges = [e.origin_dx for e in selected if e.accepted and e.direction == "horizontal"]
    vertical_edges = [e.origin_dy for e in selected if e.accepted and e.direction == "vertical"]
    widths = [t.width for t in tiles.values()]
    heights = [t.height for t in tiles.values()]
    pitch_x = int(round(np.median(horizontal_edges))) if horizontal_edges else int(round(np.median(widths) * 0.80))
    pitch_y = int(round(np.median(vertical_edges))) if vertical_edges else int(round(np.median(heights) * 0.80))
    for tile in tiles.values():
        origins.setdefault(tile.name, (tile.row * pitch_y, tile.col * pitch_x))

    min_y = min(y for y, _ in origins.values())
    min_x = min(x for _, x in origins.values())
    if min_y < 0 or min_x < 0:
        origins = {k: (y - min_y, x - min_x) for k, (y, x) in origins.items()}
    return origins


## Mosaic and Preview Writers

In [ ]:
def simple_ipf_rgb(tile: TileGrid) -> np.ndarray:
    """Fast orientation color preview; not a substitute for orix/MTEX IPF keys."""
    # Use crystal direction of sample Z in crystal frame as a stable RGB proxy.
    r = euler_to_matrix(tile.phi1, tile.Phi, tile.phi2)
    direction = np.abs(r[..., :, 2])
    direction[~tile.valid] = 0.0
    denom = np.max(direction, axis=2, keepdims=True)
    rgb = direction / np.where(denom > 1e-9, denom, 1.0)
    rgb[~tile.valid] = np.nan
    return np.clip(rgb, 0.0, 1.0)


def mosaic_tiles(tiles: Dict[str, TileGrid], origins: Dict[str, Tuple[int, int]]):
    max_y = max(origins[name][0] + tile.height for name, tile in tiles.items())
    max_x = max(origins[name][1] + tile.width for name, tile in tiles.items())
    iq = np.full((max_y, max_x), np.nan, dtype=float)
    ci = np.full((max_y, max_x), np.nan, dtype=float)
    rgb = np.full((max_y, max_x, 3), np.nan, dtype=float)
    source = np.full((max_y, max_x), "", dtype=object)

    for name, tile in sorted(tiles.items(), key=lambda kv: (kv[1].row, kv[1].col)):
        y0, x0 = origins[name]
        ys = slice(y0, y0 + tile.height)
        xs = slice(x0, x0 + tile.width)
        target_ci = ci[ys, xs]
        replace = tile.valid & (np.isnan(target_ci) | (tile.ci >= np.nan_to_num(target_ci, nan=-np.inf)))
        target_iq = iq[ys, xs]
        target_rgb = rgb[ys, xs]
        target_source = source[ys, xs]
        tile_rgb = simple_ipf_rgb(tile)
        target_iq[replace] = tile.iq[replace]
        target_ci[replace] = tile.ci[replace]
        target_rgb[replace] = tile_rgb[replace]
        target_source[replace] = name
        iq[ys, xs] = target_iq
        ci[ys, xs] = target_ci
        rgb[ys, xs] = target_rgb
        source[ys, xs] = target_source
    return iq, ci, rgb, source


def save_image(path: Path, arr: np.ndarray, cmap: str = "gray") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(10, 8))
    if arr.ndim == 3:
        shown = np.nan_to_num(arr, nan=1.0)
        ax.imshow(shown, origin="upper")
    else:
        ax.imshow(arr, cmap=cmap, origin="upper")
    ax.set_axis_off()
    fig.tight_layout(pad=0)
    fig.savefig(path, dpi=200, bbox_inches="tight", pad_inches=0)
    plt.close(fig)


## Pipeline Runner

In [ ]:
def run(args: argparse.Namespace) -> None:
    folder = Path(args.tiles).resolve()
    out_dir = Path(args.output).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    manifest = read_manifest(folder)
    if args.max_tiles:
        manifest = manifest.head(args.max_tiles).copy()

    tiles: Dict[str, TileGrid] = {}
    for row in manifest.itertuples(index=False):
        df = read_ang_numeric(Path(row.path))
        tile = gridify_ang(df, str(row.tile_name), int(row.row), int(row.col), Path(row.path))
        tiles[tile.name] = tile

    manifest_by_name = manifest.set_index("tile_name")
    pairs = adjacent_pairs(manifest)
    if args.max_pairs:
        pairs = pairs[: args.max_pairs]

    all_candidates: List[PairCandidate] = []
    for row_a, row_b, direction in pairs:
        a = tiles[str(row_a.tile_name)]
        b = tiles[str(row_b.tile_name)]
        overlap_px = infer_overlap_px(a, b, row_a, row_b, direction)
        strips_a, strips_b = strip_pair(a, b, direction, overlap_px)
        pair_id = f"{direction}_r{int(row_a.row)}_c{int(row_a.col)}__r{int(row_b.row)}_c{int(row_b.col)}"
        for sy, sx, source in candidate_shift_set(strips_a, strips_b, args.search_radius):
            cand = evaluate_candidate(
                pair_id,
                direction,
                a,
                b,
                strips_a,
                strips_b,
                sy,
                sx,
                source,
                overlap_px,
                args.orientation_samples,
            )
            if cand is not None:
                all_candidates.append(cand)

    model, scaler, feature_cols, self_supervised_report = train_ml_scorer(
        all_candidates,
        use_training=args.use_ml_training,
        train_fraction=args.self_supervised_train_fraction,
    )
    selected = choose_best_candidates(
        all_candidates,
        min_accept_score=args.min_accept_score,
        max_accept_misorientation=args.max_accept_misorientation,
        min_valid_fraction=args.min_valid_fraction,
    )
    origins = solve_tile_origins(tiles, selected)
    iq_mosaic, ci_mosaic, rgb_mosaic, source_mosaic = mosaic_tiles(tiles, origins)

    cand_df = pd.DataFrame([c.__dict__ for c in all_candidates])
    selected_df = pd.DataFrame([c.__dict__ for c in selected])
    origins_df = pd.DataFrame(
        [{"tile_name": name, "origin_y_px": y, "origin_x_px": x, "row": tiles[name].row, "col": tiles[name].col} for name, (y, x) in origins.items()]
    ).sort_values(["row", "col"])

    cand_df.to_csv(out_dir / "candidate_shift_scores.csv", index=False)
    selected_df.to_csv(out_dir / "selected_pair_shifts.csv", index=False)
    origins_df.to_csv(out_dir / "tile_origins.csv", index=False)
    np.savez_compressed(out_dir / "stitched_mosaic_arrays.npz", iq=iq_mosaic, ci=ci_mosaic, ipf_rgb=rgb_mosaic, source=source_mosaic)
    save_image(out_dir / "stitched_iq.png", iq_mosaic, cmap="gray")
    save_image(out_dir / "stitched_ci.png", ci_mosaic, cmap="viridis")
    save_image(out_dir / "stitched_ipf_preview.png", rgb_mosaic)

    summary = {
        "tiles_folder": str(folder),
        "output_dir": str(out_dir),
        "n_tiles": len(tiles),
        "n_pairs": len(pairs),
        "n_candidates": len(all_candidates),
        "n_accepted_seams": int(sum(c.accepted for c in selected)),
        "ml_model": "logistic_regression_self_supervised" if model is not None else "classical_score_fallback",
        "feature_columns": feature_cols,
        "self_supervised_training": self_supervised_report,
        "selected_shift_summary": {
            "median_iq_corr": float(selected_df["iq_corr"].median()) if not selected_df.empty else None,
            "median_ci_corr": float(selected_df["ci_corr"].median()) if not selected_df.empty else None,
            "median_misorientation_deg": float(selected_df["median_misorientation_deg"].median()) if not selected_df.empty else None,
            "p90_misorientation_deg": float(selected_df["median_misorientation_deg"].quantile(0.90)) if not selected_df.empty else None,
            "median_valid_fraction": float(selected_df["valid_fraction"].median()) if not selected_df.empty else None,
            "accepted_seam_fraction": float(selected_df["accepted"].mean()) if not selected_df.empty else None,
        },
    }
    if model is not None:
        summary["ml_coefficients"] = {
            col: float(coef) for col, coef in zip(feature_cols, model.coef_[0])
        }
        summary["ml_intercept"] = float(model.intercept_[0])

    (out_dir / "stitch_summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))


## Where the Self-Supervision Is

No manual seam labels are used. The notebook generates pseudo-labels from EBSD physics: a candidate shift is a pseudo-positive when its overlap has enough valid pixels, high IQ agreement, and low FCC symmetry-aware disorientation. Candidate shifts that do not satisfy those checks act as pseudo-negatives. The logistic model trains on one subset of right/down seam pairs and validates on held-out seam pairs.

For another single-phase FCC material, keep the same cubic symmetry setting and point `TILES_FOLDER` to that material's clean cropped ANG tiles. If the material is multiphase or non-FCC, change the phase/symmetry section before running.


## Configure Run

In [ ]:
from types import SimpleNamespace

# SCC 316 clean cropped tiles. Change this if you want a different folder.
TILES_FOLDER = "Cropped"
OUTPUT_DIR = "raster_ml_stitch_output"

args = SimpleNamespace(
    tiles=TILES_FOLDER,
    output=OUTPUT_DIR,
    search_radius=1,
    orientation_samples=120,
    max_tiles=0,      # set >0 for quick debug
    max_pairs=0,      # set >0 for quick debug
    use_ml_training=True,
    self_supervised_train_fraction=0.70,
    min_accept_score=0.65,
    max_accept_misorientation=8.0,
    min_valid_fraction=0.60,
)
args


## Run Stitching

In [ ]:
run(args)


## Validation: Right/Down Seams, Calibration, and Placement

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)


def finite_float(value):
    if value is None:
        return None
    try:
        value = float(value)
    except Exception:
        return value
    return value if np.isfinite(value) else None


def binary_metrics(y_true, y_score, threshold=0.65):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    out = {
        'threshold': threshold,
        'accuracy': (tp + tn) / max(tp + tn + fp + fn, 1),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
        'false_accept_rate': fp / max(fp + tn, 1),
        'missed_accept_rate': fn / max(fn + tp, 1),
    }
    if len(np.unique(y_true)) > 1:
        out['roc_auc'] = roc_auc_score(y_true, y_score)
        out['pr_auc'] = average_precision_score(y_true, y_score)
    else:
        out['roc_auc'] = None
        out['pr_auc'] = None
    return {k: finite_float(v) for k, v in out.items()}


def expected_calibration_error(y_true, y_score, n_bins=10):
    y_true = np.asarray(y_true).astype(float)
    y_score = np.asarray(y_score).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    rows = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_score >= lo) & (y_score < hi if hi < 1 else y_score <= hi)
        if not mask.any():
            rows.append({'bin_left': lo, 'bin_right': hi, 'count': 0, 'mean_score': np.nan, 'empirical_rate': np.nan})
            continue
        mean_score = float(y_score[mask].mean())
        empirical = float(y_true[mask].mean())
        weight = float(mask.mean())
        ece += weight * abs(empirical - mean_score)
        rows.append({'bin_left': lo, 'bin_right': hi, 'count': int(mask.sum()), 'mean_score': mean_score, 'empirical_rate': empirical})
    return float(ece), pd.DataFrame(rows)


def validate_manifest_layout(output_dir, tiles_folder):
    output_dir = Path(output_dir)
    tiles_folder = Path(tiles_folder)
    manifest_path = tiles_folder / 'tile_manifest.csv'
    origins_path = output_dir / 'tile_origins.csv'
    if not manifest_path.exists() or not origins_path.exists():
        return {}

    manifest = pd.read_csv(manifest_path)
    manifest = manifest[manifest.tile_name.astype(str).str.endswith('_clean.ang')].copy()
    if 'distorted' in manifest:
        manifest = manifest[manifest.distorted.astype(str).str.lower().isin(['false', '0', 'no'])]
    origins = pd.read_csv(origins_path)
    if manifest.empty or origins.empty:
        return {}

    first_tile = tiles_folder / str(manifest.iloc[0].tile_name)
    first_grid = gridify_ang(read_ang_numeric(first_tile), first_tile.name, int(manifest.iloc[0].row), int(manifest.iloc[0].col), first_tile)
    min_x = float(manifest.xmin.min()) if 'xmin' in manifest else 0.0
    min_y = float(manifest.ymin.min()) if 'ymin' in manifest else 0.0
    manifest['expected_origin_x_px'] = ((manifest.xmin.astype(float) - min_x) / max(first_grid.x_step, 1e-9)).round().astype(int)
    manifest['expected_origin_y_px'] = ((manifest.ymin.astype(float) - min_y) / max(first_grid.y_step, 1e-9)).round().astype(int)
    merged = origins.merge(manifest[['tile_name', 'expected_origin_x_px', 'expected_origin_y_px']], on='tile_name', how='inner')
    merged['origin_dx_error_px'] = merged.origin_x_px - merged.expected_origin_x_px
    merged['origin_dy_error_px'] = merged.origin_y_px - merged.expected_origin_y_px
    merged.to_csv(output_dir / 'manifest_layout_validation.csv', index=False)
    return {
        'n_tiles_compared': int(len(merged)),
        'x_step': finite_float(first_grid.x_step),
        'y_step': finite_float(first_grid.y_step),
        'mean_abs_origin_dx_error_px': finite_float(merged.origin_dx_error_px.abs().mean()),
        'mean_abs_origin_dy_error_px': finite_float(merged.origin_dy_error_px.abs().mean()),
        'max_abs_origin_dx_error_px': finite_float(merged.origin_dx_error_px.abs().max()),
        'max_abs_origin_dy_error_px': finite_float(merged.origin_dy_error_px.abs().max()),
    }


def validate_raster_stitch_outputs(output_dir=OUTPUT_DIR, tiles_folder=TILES_FOLDER):
    output_dir = Path(output_dir)
    validation_dir = output_dir / 'validation'
    validation_dir.mkdir(parents=True, exist_ok=True)

    candidates = pd.read_csv(output_dir / 'candidate_shift_scores.csv')
    selected = pd.read_csv(output_dir / 'selected_pair_shifts.csv')
    origins = pd.read_csv(output_dir / 'tile_origins.csv')

    # Right/down seam validation, matching the old notebook's right+down intent.
    direction_summary = selected.groupby('direction').agg(
        n_pairs=('pair_id', 'size'),
        accepted=('accepted', 'sum'),
        acceptance_rate=('accepted', 'mean'),
        median_misorientation_deg=('median_misorientation_deg', 'median'),
        p90_misorientation_deg=('median_misorientation_deg', lambda s: s.quantile(0.90)),
        median_iq_corr=('iq_corr', 'median'),
        median_ci_corr=('ci_corr', 'median'),
        median_final_score=('final_score', 'median'),
    ).reset_index()
    direction_summary.to_csv(validation_dir / 'direction_summary.csv', index=False)

    # Pseudo-label diagnostics for the ML seam scorer.
    score_metrics = {
        score_col: binary_metrics(candidates.pseudo_label, candidates[score_col], threshold=args.min_accept_score)
        for score_col in ['classical_score', 'ml_probability', 'final_score']
        if score_col in candidates
    }
    ece, calibration_table = expected_calibration_error(candidates.pseudo_label, candidates.final_score, n_bins=10)
    calibration_table.to_csv(validation_dir / 'calibration_table.csv', index=False)

    # Confusion matrix for final candidate score against self-supervised labels.
    y_true = candidates.pseudo_label.astype(int).to_numpy()
    y_pred = (candidates.final_score >= args.min_accept_score).astype(int).to_numpy()
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap='Blues')
    for (i, j), val in np.ndenumerate(cm):
        ax.text(j, i, str(val), ha='center', va='center')
    ax.set_xticks([0, 1], labels=['Pred 0', 'Pred 1'])
    ax.set_yticks([0, 1], labels=['Pseudo 0', 'Pseudo 1'])
    ax.set_title('Final score confusion matrix')
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(validation_dir / 'confusion_matrix_final_score.png', dpi=180)
    plt.show()

    # Calibration curve.
    valid_bins = calibration_table[calibration_table['count'] > 0]
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(valid_bins.mean_score, valid_bins.empirical_rate, 'o-', label=f'ECE={ece:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set_xlabel('Mean final score')
    ax.set_ylabel('Empirical pseudo-positive rate')
    ax.set_title('Self-supervised seam score calibration')
    ax.legend()
    fig.tight_layout()
    fig.savefig(validation_dir / 'calibration_curve_final_score.png', dpi=180)
    plt.show()

    # Right/down acceptance and misorientation plots.
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(direction_summary.direction, direction_summary.acceptance_rate)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Accepted seam fraction')
    ax.set_title('Right/down seam acceptance')
    fig.tight_layout()
    fig.savefig(validation_dir / 'seam_acceptance_by_direction.png', dpi=180)
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4))
    selected.boxplot(column='median_misorientation_deg', by='direction', ax=ax)
    ax.set_title('Selected seam median FCC disorientation')
    ax.set_xlabel('Direction')
    ax.set_ylabel('Degrees')
    fig.suptitle('')
    fig.tight_layout()
    fig.savefig(validation_dir / 'misorientation_by_direction.png', dpi=180)
    plt.show()

    # Placement residuals: accepted seam vector versus solved tile origins.
    origin_lookup = origins.set_index('tile_name')[['origin_y_px', 'origin_x_px']].to_dict('index')
    residual_rows = []
    for rec in selected.itertuples(index=False):
        if not bool(rec.accepted):
            continue
        a = origin_lookup.get(rec.tile_a)
        b = origin_lookup.get(rec.tile_b)
        if a is None or b is None:
            continue
        residual_rows.append({
            'pair_id': rec.pair_id,
            'direction': rec.direction,
            'residual_y_px': (b['origin_y_px'] - a['origin_y_px']) - rec.origin_dy,
            'residual_x_px': (b['origin_x_px'] - a['origin_x_px']) - rec.origin_dx,
        })
    residual_df = pd.DataFrame(residual_rows)
    residual_df.to_csv(validation_dir / 'placement_residuals.csv', index=False)
    if not residual_df.empty:
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.scatter(residual_df.residual_x_px, residual_df.residual_y_px, s=12)
        ax.axhline(0, color='k', linewidth=1)
        ax.axvline(0, color='k', linewidth=1)
        ax.set_xlabel('x residual px')
        ax.set_ylabel('y residual px')
        ax.set_title('Accepted seam placement residuals')
        fig.tight_layout()
        fig.savefig(validation_dir / 'placement_residuals.png', dpi=180)
        plt.show()

    manifest_validation = validate_manifest_layout(output_dir, tiles_folder)

    validation_summary = {
        'n_candidates': int(len(candidates)),
        'n_selected_pairs': int(len(selected)),
        'n_accepted_seams': int(selected.accepted.sum()),
        'accepted_seam_fraction': finite_float(selected.accepted.mean()),
        'direction_summary': direction_summary.to_dict('records'),
        'score_metrics': score_metrics,
        'calibration': {'ECE': finite_float(ece)},
        'placement_residuals': {
            'n_edges': int(len(residual_df)),
            'mean_abs_x_px': finite_float(residual_df.residual_x_px.abs().mean()) if not residual_df.empty else None,
            'mean_abs_y_px': finite_float(residual_df.residual_y_px.abs().mean()) if not residual_df.empty else None,
            'max_abs_x_px': finite_float(residual_df.residual_x_px.abs().max()) if not residual_df.empty else None,
            'max_abs_y_px': finite_float(residual_df.residual_y_px.abs().max()) if not residual_df.empty else None,
        },
        'manifest_layout_validation': manifest_validation,
        'note': 'Pseudo-label metrics are self-supervised seam diagnostics, not external ground truth. Manifest validation checks consistency with cropped parent tile coordinates.',
    }
    (validation_dir / 'validation_summary.json').write_text(json.dumps(validation_summary, indent=2))
    print(json.dumps(validation_summary, indent=2))
    return validation_summary, direction_summary, residual_df


validation_summary, direction_summary, placement_residuals = validate_raster_stitch_outputs()


## Inspect Results

In [ ]:
from IPython.display import Image, display
import json
from pathlib import Path
import pandas as pd

out = Path(OUTPUT_DIR)
summary = json.loads((out / "stitch_summary.json").read_text())
print(json.dumps(summary, indent=2))

selected = pd.read_csv(out / "selected_pair_shifts.csv")
display(selected[[
    "pair_id", "direction", "accepted", "shift_y", "shift_x",
    "origin_dy", "origin_dx", "iq_corr", "ci_corr",
    "median_misorientation_deg", "ml_probability", "final_score"
]].sort_values(["accepted", "median_misorientation_deg"], ascending=[True, False]).head(25))

for image_name in ["stitched_iq.png", "stitched_ci.png", "stitched_ipf_preview.png"]:
    print(image_name)
    display(Image(filename=str(out / image_name)))
